In [1]:
import pandas as pd, numpy as np
d = pd.read_csv("nifty50_features.csv", parse_dates=["Date"])

d["log_ret"] = np.log(d["Close"]).diff()
d["vol_5d"]  = d["log_ret"].rolling(5).std() * np.sqrt(252) * 100
d["abs_ret"] = d["ret"].abs()
# volatility over the NEXT 5 trading days (annualised %)
d["fut_vol_5d"] = d["log_ret"].rolling(5).std().shift(-5) * np.sqrt(252) * 100

# only information known at NIFTY's close (no S&P/crude/FX moves from that night)
vfeatures = ["vol_5d", "vol_21d", "vix", "vix_chg", "abs_ret", "ret",
             "ret_lag1", "ret_lag2", "ret_lag3", "ret_lag5",
             "ma10_ratio", "ma50_ratio", "rsi14", "dow"]

dv = d.dropna(subset=vfeatures + ["fut_vol_5d"]).reset_index(drop=True)

train_v = dv[dv["Date"] < "2019-12-20"]     # small gap so target windows don't overlap validation
val_v   = dv[(dv["Date"] >= "2020-01-01") & (dv["Date"] < "2023-01-01")]
test_v  = dv[dv["Date"] >= "2023-01-01"]    # untouched until the end
print(len(train_v), len(val_v), len(test_v))

2382 746 926


In [3]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

y = val_v["fut_vol_5d"]
rows = [
    {"model": "Baseline: current 21-day vol", "MAE": mean_absolute_error(y, val_v["vol_21d"]), "R2": r2_score(y, val_v["vol_21d"])},
    {"model": "Baseline: current India VIX",  "MAE": mean_absolute_error(y, val_v["vix"]),     "R2": r2_score(y, val_v["vix"])},
]

models = {
    "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=10)),
    "Random Forest": RandomForestRegressor(n_estimators=300, max_depth=5, min_samples_leaf=30, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=200, max_depth=2, learning_rate=0.05, subsample=0.8, random_state=42),
}
for name, m in models.items():
    m.fit(train_v[vfeatures], train_v["fut_vol_5d"])
    p = m.predict(val_v[vfeatures])
    rows.append({"model": name, "MAE": mean_absolute_error(y, p), "R2": r2_score(y, p)})

print(pd.DataFrame(rows).round(3))

                          model    MAE     R2
0  Baseline: current 21-day vol  7.933  0.235
1   Baseline: current India VIX  8.506  0.423
2                         Ridge  6.376  0.518
3                 Random Forest  6.949  0.244
4             Gradient Boosting  6.852  0.318
